In [ ]:
---------------------------------------------------------------------
-- Exercise 1 Setup
-- Create a dbo.Customers staging table to work with
---------------------------------------------------------------------
 
DROP TABLE IF EXISTS dbo.Customers;
 
CREATE TABLE dbo.Customers
(
  CustomerId          INT           NOT NULL PRIMARY KEY,
  CustomerCompanyName NVARCHAR(40)  NOT NULL,
  CustomerCountry     NVARCHAR(15)  NOT NULL,
  CustomerRegion      NVARCHAR(15)  NULL,
  CustomerCity        NVARCHAR(15)  NOT NULL
);


In [ ]:
---------------------------------------------------------------------
-- Exercise 1-1
-- Insert a single row into dbo.Customers with:
-- CustomerId:          100
-- CustomerCompanyName: Coho Winery
-- CustomerCountry:     USA
-- CustomerRegion:      WA
-- CustomerCity:        Redmond
---------------------------------------------------------------------
 
INSERT INTO dbo.Customers (CustomerId, CustomerCompanyName, CustomerCountry, CustomerRegion, CustomerCity)
VALUES (100, N'Coho Winery', N'USA', N'WA', N'Redmond');
 
SELECT * FROM dbo.Customers;

In [ ]:
---------------------------------------------------------------------
-- Exercise 1-2
-- Insert into dbo.Customers all customers from Sales.Customer
-- who have placed at least one order
---------------------------------------------------------------------
 
INSERT INTO dbo.Customers (CustomerId, CustomerCompanyName, CustomerCountry, CustomerRegion, CustomerCity)
  SELECT C.CustomerId, C.CustomerCompanyName, C.CustomerCountry, 
         C.CustomerRegion, C.CustomerCity
  FROM Sales.Customer AS C
  WHERE EXISTS
    (SELECT * FROM Sales.[Order] AS O
     WHERE O.CustomerId = C.CustomerId);
 
SELECT * FROM dbo.Customers;

In [ ]:
---------------------------------------------------------------------
-- Exercise 1-3
-- Use SELECT INTO to create and populate dbo.Orders
-- with orders placed in the years 2014 through 2016
---------------------------------------------------------------------
 
DROP TABLE IF EXISTS dbo.Orders;
 
SELECT *
INTO dbo.Orders
FROM Sales.[Order]
WHERE OrderDate >= '20140101'
  AND OrderDate < '20170101';
 
SELECT * FROM dbo.Orders;

In [ ]:
---------------------------------------------------------------------
-- Exercise 2
-- Delete from dbo.Orders all orders placed before August 2015
-- Use OUTPUT to return the OrderId and OrderDate of deleted rows
---------------------------------------------------------------------
 
DELETE FROM dbo.Orders
  OUTPUT deleted.OrderId, deleted.OrderDate
WHERE OrderDate < '20150801';

In [ ]:
---------------------------------------------------------------------
-- Exercise 3
-- Delete from dbo.Orders all orders placed by customers from Germany
---------------------------------------------------------------------
 
DELETE FROM dbo.Orders
WHERE EXISTS
  (SELECT *
   FROM dbo.Customers AS C
   WHERE dbo.Orders.CustomerId = C.CustomerId
     AND C.CustomerCountry = N'Germany');
 
SELECT * FROM dbo.Orders;

In [ ]:
---------------------------------------------------------------------
-- Exercise 4
-- Update dbo.Customers and change all NULL CustomerRegion values
-- to '<None>' -- use OUTPUT to show CustomerId, old and new region
---------------------------------------------------------------------
 
UPDATE dbo.Customers
  SET CustomerRegion = '<None>'
OUTPUT
  deleted.CustomerId,
  deleted.CustomerRegion AS OldRegion,
  inserted.CustomerRegion AS NewRegion
WHERE CustomerRegion IS NULL;

In [ ]:
---------------------------------------------------------------------
-- Exercise 5
-- Update dbo.Orders for all orders placed by customers from the UK
-- Set their shipping country, region, and city to match
-- the corresponding customer's values from dbo.Customers
---------------------------------------------------------------------
 
UPDATE O
  SET O.ShipToAddress = C.CustomerCity
FROM dbo.Orders AS O
  INNER JOIN dbo.Customers AS C
    ON O.CustomerId = C.CustomerId
WHERE C.CustomerCountry = N'UK';
 
SELECT * FROM dbo.Orders;

In [ ]:
---------------------------------------------------------------------
-- Exercise 6
-- Create dbo.Orders and dbo.OrderDetails, populate them,
-- then truncate both tables handling the FK constraint
---------------------------------------------------------------------
 
DROP TABLE IF EXISTS dbo.OrderDetails, dbo.Orders;
 
CREATE TABLE dbo.Orders
(
  OrderId       INT           NOT NULL,
  CustomerId    INT           NULL,
  EmployeeId    INT           NOT NULL,
  OrderDate     DATE          NOT NULL,
  RequiredDate  DATE          NOT NULL,
  ShipToDate    DATE          NULL,
  ShipperId     INT           NOT NULL,
  Freight       MONEY         NOT NULL
    CONSTRAINT DFT_Orders_Freight DEFAULT(0),
  ShipToName    NVARCHAR(40)  NOT NULL,
  ShipToAddress NVARCHAR(60)  NOT NULL,
  CONSTRAINT PK_Orders PRIMARY KEY(OrderId)
);
 
CREATE TABLE dbo.OrderDetails
(
  OrderId             INT           NOT NULL,
  ProductId           INT           NOT NULL,
  UnitPrice           MONEY         NOT NULL
    CONSTRAINT DFT_OrderDetails_UnitPrice DEFAULT(0),
  Quantity            SMALLINT      NOT NULL
    CONSTRAINT DFT_OrderDetails_Qty DEFAULT(1),
  DiscountPercentage  NUMERIC(4,3)  NOT NULL
    CONSTRAINT DFT_OrderDetails_Discount DEFAULT(0),
  CONSTRAINT PK_OrderDetails PRIMARY KEY(OrderId, ProductId),
  CONSTRAINT FK_OrderDetails_Orders FOREIGN KEY(OrderId)
    REFERENCES dbo.Orders(OrderId),
  CONSTRAINT CHK_Discount   CHECK (DiscountPercentage BETWEEN 0 AND 1),
  CONSTRAINT CHK_Quantity   CHECK (Quantity > 0),
  CONSTRAINT CHK_UnitPrice  CHECK (UnitPrice >= 0)
);
GO
 
INSERT INTO dbo.Orders
  SELECT OrderId, CustomerId, EmployeeId, OrderDate, RequiredDate,
         ShipToDate, ShipperId, Freight, ShipToName, ShipToAddress
  FROM Sales.[Order];
 
INSERT INTO dbo.OrderDetails
  SELECT OrderId, ProductId, UnitPrice, Quantity, DiscountPercentage
  FROM Sales.OrderDetail;
 
-- Truncate both tables (must drop FK first, then re-add)
ALTER TABLE dbo.OrderDetails DROP CONSTRAINT FK_OrderDetails_Orders;
 
TRUNCATE TABLE dbo.OrderDetails;
TRUNCATE TABLE dbo.Orders;
 
ALTER TABLE dbo.OrderDetails ADD CONSTRAINT FK_OrderDetails_Orders
  FOREIGN KEY(OrderId) REFERENCES dbo.Orders(OrderId);

Propositions:

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

<span style="color: #5c6370;font-style: italic;">-- P1: INSERT VALUES</span>

<span style="color: #5c6370;font-style: italic;">-- A new supplier needs to be added to the database before their</span>

<span style="color: #5c6370;font-style: italic;">-- contract is finalized. We insert their info directly as a single row.</span>

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

In [ ]:
INSERT INTO Production.Supplier (
    SupplierCompanyName, SupplierContactName, SupplierContactTitle,
    SupplierAddress, SupplierCity, SupplierCountry, SupplierPhoneNumber
)
VALUES (
    'Brooklyn Fresh Co.', 'John Smith', 'Sales Manager',
    '123 Atlantic Ave', 'New York', 'USA', '718-555-0199'
);

SELECT * FROM Production.Supplier;

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

<span style="color: #5c6370;font-style: italic;">-- P2: INSERT SELECT (SELECT INTO)</span>

<span style="color: #5c6370;font-style: italic;">-- We want to identify high-value customers who spent over $10,000</span>

<span style="color: #5c6370;font-style: italic;">-- and move them into a separate VIP table for a loyalty campaign.</span>

<span style="color: #5c6370;font-style: italic;">-- This uses SELECT INTO to create and populate the table in one step.</span>

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

In [ ]:
DROP TABLE IF EXISTS dbo.VIPCustomers;

SELECT C.CustomerId, C.CustomerCompanyName, C.CustomerCountry,
       SUM(OD.UnitPrice * OD.Quantity) AS TotalSpent
INTO dbo.VIPCustomers
FROM Sales.Customer C
JOIN Sales.[Order] O ON C.CustomerId = O.CustomerId
JOIN Sales.OrderDetail OD ON O.OrderId = OD.OrderId
GROUP BY C.CustomerId, C.CustomerCompanyName, C.CustomerCountry
HAVING SUM(OD.UnitPrice * OD.Quantity) > 10000;

SELECT * FROM dbo.VIPCustomers;

  

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

<span style="color: #5c6370;font-style: italic;">-- P3: SELECT INTO</span>

<span style="color: #5c6370;font-style: italic;">-- Before removing discontinued products from the active catalog,</span>

<span style="color: #5c6370;font-style: italic;">-- we archive them into a separate table as a backup.</span>

<span style="color: #5c6370;font-style: italic;">-- SELECT INTO creates and populates the archive table in one shot.</span>

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

In [ ]:
DROP TABLE IF EXISTS dbo.DiscontinuedProducts;

SELECT *
INTO dbo.DiscontinuedProducts
FROM Production.Product
WHERE Discontinued = 1;

SELECT * FROM dbo.DiscontinuedProducts;

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

<span style="color: #5c6370;font-style: italic;">-- P4: UPDATE</span>

<span style="color: #5c6370;font-style: italic;">-- Supplier costs went up, so Beverage prices need a 10% increase.</span>

<span style="color: #5c6370;font-style: italic;">-- We update all products in the Beverages category with a single UPDATE.</span>

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

In [ ]:

UPDATE Production.Product
SET UnitPrice = UnitPrice * 1.10
WHERE CategoryId = 1;

SELECT * FROM Production.Product WHERE CategoryId = 1;

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

<span style="color: #5c6370;font-style: italic;">-- P5: UPDATE with JOIN</span>

<span style="color: #5c6370;font-style: italic;">-- Some products may belong to a supplier whose info recently changed.</span>

<span style="color: #5c6370;font-style: italic;">-- We update the product's SupplierId by joining to Production.Supplier</span>

<span style="color: #5c6370;font-style: italic;">-- to make sure the link is correct for a specific company.</span>

<span style="color: #5c6370;font-style: italic;">--------------------------------------------------------------------</span>

In [ ]:
UPDATE P
SET P.SupplierId = S.SupplierId
FROM Production.Product P
JOIN Production.Supplier S ON P.SupplierId = S.SupplierId
WHERE S.SupplierCompanyName = 'Brooklyn Fresh Co.';

SELECT * FROM Production.Supplier;
SELECT * FROM Production.Product WHERE CategoryId = 1;

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

<span style="color: #5c6370;font-style: italic;">-- P6: UPDATE with OUTPUT</span>

<span style="color: #5c6370;font-style: italic;">-- We raise Seafood product prices by 5% and use OUTPUT to capture</span>

<span style="color: #5c6370;font-style: italic;">-- both the old and new price side by side. Useful for auditing</span>

<span style="color: #5c6370;font-style: italic;">-- what changed before committing to the update.</span>

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

In [ ]:
UPDATE Production.Product
SET UnitPrice = UnitPrice * 1.05
OUTPUT
    inserted.ProductId,
    deleted.UnitPrice AS OldPrice,
    inserted.UnitPrice AS NewPrice
WHERE CategoryId = 8;

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

<span style="color: #5c6370;font-style: italic;">-- P7: DELETE with OUTPUT</span>

<span style="color: #5c6370;font-style: italic;">-- We need to clean out old orders from the database.</span>

<span style="color: #5c6370;font-style: italic;">-- OUTPUT lets us capture the deleted records before they're gone,</span>

<span style="color: #5c6370;font-style: italic;">-- which is useful for compliance or logging purposes.</span>

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

In [ ]:
DELETE FROM Sales.[Order]
OUTPUT
    deleted.OrderId,
    deleted.CustomerId,
    deleted.OrderDate
WHERE OrderDate < '19970101';

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

<span style="color: #5c6370;font-style: italic;">-- P8: DELETE with subquery</span>

<span style="color: #5c6370;font-style: italic;">-- Over time, some orders end up referencing customers that no</span>

<span style="color: #5c6370;font-style: italic;">-- longer exist in the system. This query finds and removes those</span>

<span style="color: #5c6370;font-style: italic;">-- orphaned orders using a subquery to check for missing customers.</span>

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

In [ ]:
---------------------------------------------------------------------
-- P8: DELETE with subquery
-- Over time, some orders end up referencing customers that no
-- longer exist in the system. This query finds and removes those
-- orphaned orders using a subquery to check for missing customers.
---------------------------------------------------------------------

DELETE FROM Sales.[Order]
WHERE CustomerId NOT IN (
    SELECT CustomerId FROM Sales.Customer
);

SELECT COUNT(*) FROM Sales.[Order];

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

<span style="color: #5c6370;font-style: italic;">-- P9: MERGE (Upsert)</span>

<span style="color: #5c6370;font-style: italic;">-- Shipper contact info changes and new shippers get added over time.</span>

<span style="color: #5c6370;font-style: italic;">-- MERGE handles both updates and inserts in one clean statement.</span>

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

In [ ]:

DROP TABLE IF EXISTS dbo.ShipperStage;

CREATE TABLE dbo.ShipperStage (
    ShipperId          INT,
    ShipperCompanyName NVARCHAR(40),
    PhoneNumber        NVARCHAR(24)
);

INSERT INTO dbo.ShipperStage VALUES
    (1,  'Speedy Express Updated', '503-555-9999'),
    (99, 'Fast Ship Co.',          '718-555-0101');

MERGE INTO Sales.Shipper AS TGT
USING dbo.ShipperStage AS SRC
    ON TGT.ShipperId = SRC.ShipperId
WHEN MATCHED THEN
    UPDATE SET TGT.ShipperCompanyName = SRC.ShipperCompanyName,
               TGT.PhoneNumber = SRC.PhoneNumber
WHEN NOT MATCHED THEN
    INSERT (ShipperCompanyName, PhoneNumber)
    VALUES (SRC.ShipperCompanyName, SRC.PhoneNumber);

SELECT * FROM Sales.Shipper;

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

<span style="color: #5c6370;font-style: italic;">-- P10: Nested DML</span>

<span style="color: #5c6370;font-style: italic;">-- We raise prices for Supplier 1's products by 15% and only log</span>

<span style="color: #5c6370;font-style: italic;">-- the ones that crossed the $20 mark into an audit table.</span>

<span style="color: #5c6370;font-style: italic;">-- This uses a nested DML pattern where UPDATE OUTPUT feeds an INSERT.</span>

<span style="color: #5c6370;font-style: italic;">---------------------------------------------------------------------</span>

In [ ]:

DROP TABLE IF EXISTS dbo.ProductsAudit;

CREATE TABLE dbo.ProductsAudit (
    LSN       INT IDENTITY PRIMARY KEY,
    TS        DATETIME2 DEFAULT SYSDATETIME(),
    ProductId INT,
    OldPrice  MONEY,
    NewPrice  MONEY
);

INSERT INTO dbo.ProductsAudit (ProductId, OldPrice, NewPrice)
    SELECT ProductId, OldPrice, NewPrice
    FROM (
        UPDATE Production.Product
            SET UnitPrice = UnitPrice * 1.15
        OUTPUT
            inserted.ProductId,
            deleted.UnitPrice AS OldPrice,
            inserted.UnitPrice AS NewPrice
        WHERE SupplierId = 1
    ) AS D
    WHERE OldPrice < 20.0 AND NewPrice >= 20.0;

SELECT * FROM dbo.ProductsAudit;


In [ ]:
---------------------------------------------------------------------
-- Cleanup 
---------------------------------------------------------------------

DROP TABLE IF EXISTS dbo.VIPCustomers, dbo.DiscontinuedProducts,
dbo.CustomersStage, dbo.ProductsAudit;